# Part 2 – Air Quality: 01 Ingest

## Comprehensive Module Overview

Welcome to the **Ingestion Layer** of the Part 2 Air Quality data pipeline. This notebook is strictly responsible for **extracting raw data from external APIs and landing it onto the local filesystem**.

In any robust data engineering environment, it is critical to separate data extraction from data transformation. By fetching the data once and saving it as immutable raw files, we build an **idempotent** pipeline. You can rerun the downstream cleaning and EDA notebooks an infinite number of times without repeatedly hitting API rate limits or suffering from shifting upstream data over time.

### Study Period & Data Shape
We are focusing on the first quarter of 2025: **`2025-01-01` to `2025-03-31`** (a 90-day window). We capture 90 daily pollution summaries and 2,160 hourly weather snapshots per city, resulting in exactly six JSON payloads (3 cities × 2 sources).


## Environment Setup & Best Practices

This notebook intentionally minimizes external dependencies to maximize portability. We need only the built-in `json`, `pathlib`, and `time` modules, alongside the ubiquitous `requests` library for handling HTTP protocols.

```python
import json
import time
from pathlib import Path
import requests
```

> **Technical Note - Rate Limiting**: Making API requests in a tight loop without spacing them out often results in temporary IP bans (HTTP 429 Too Many Requests). We use `time.sleep()` to act as a rudimentary throttle between our network calls, ensuring we act as a polite API client.


In [1]:
import json
import os
import time
import requests
from pathlib import Path

## Configuration & Constant Definition

Here we set up our file paths using `pathlib.Path`. `pathlib` is preferred over `os.path` because it provides an object-oriented approach to file-system paths that works seamlessly across varying operating systems.

Notice how we define `ROOT` and build `DATA_DIR` from it. We then call `.mkdir(parents=True, exist_ok=True)` which guarantees our directory tree is ready to accept files. The variables `DATE_FROM` and `DATE_TO` are pinned to our 90-day Q1 window.


In [2]:
# ── Paths ────────────────────────────────────────────────────────────────────
ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'part2_airquality' else Path(os.getcwd())
RAW_AQ   = ROOT / 'data' / 'raw' / 'openaq'
RAW_WX   = ROOT / 'data' / 'raw' / 'openmeteo'
RAW_AQ.mkdir(parents=True, exist_ok=True)
RAW_WX.mkdir(parents=True, exist_ok=True)

DATE_FROM = '2025-01-01'
DATE_TO   = '2025-03-31'

## 1. OpenAQ v3 API – Daily PM2.5 Measurements

We are querying the `v3` API, targeting the `/v3/sensors/{sensor_id}/measurements` endpoint.

### API Parameterization & Data Structure
By passing `period_name=day`, we instruct the OpenAQ API backend to pre-aggregate its high-frequency readings and provide us with a single JSON object per calendar day. This drastically reduces the network payload compared to pulling raw hourly data. The JSON object crucially contains:
1. `summary.avg`: The arithmetic mean of the day's readings.
2. `summary.q75` & `summary.q98`: Intra-day percentiles. Since OpenAQ doesn't provide the 90th percentile by default, fetching these upper-percentiles will allow us to mathematically interpolate $P_{90}$ in the next notebook.
3. `coverage.observedCount`: Acts as a data-quality flag to verify whether a 'daily average' is composed of 24 hours of data, or just a few sparse measurements.


In [3]:
# OpenAQ v3 – city → sensor_id mapping for PM2.5
# Sensor IDs obtained by querying /v3/locations?city=<city>&country_id=US&parameter=pm25
OPENAQ_BASE = 'https://api.openaq.org/v3'

CITIES_AQ = {
    'nyc':     {'sensor_id': 3140,  'label': 'New York City'},
    'la':      {'sensor_id': 4913,  'label': 'Los Angeles'},
    'chicago': {'sensor_id': 12863, 'label': 'Chicago'},
}

HEADERS = {'accept': 'application/json'}

In [4]:
def fetch_openaq_daily(sensor_id: int, date_from: str, date_to: str) -> dict:
    """Fetch daily PM2.5 measurements from OpenAQ v3 for a single sensor."""
    url = f'{OPENAQ_BASE}/sensors/{sensor_id}/measurements'
    params = {
        'period_name': 'day',
        'date_from':   f'{date_from}T00:00:00Z',
        'date_to':     f'{date_to}T23:59:59Z',
        'limit':       500,
    }
    resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json()

### Idempotent Network Ingestion Loop (PM2.5)

We use a `for` loop to iterate through our `CITIES_AQ` dictionary. For each city, we construct the target path: `data/raw/openaq/pm25_<city>_2025Q1.json`.

**The Idempotency Check**: `if fp.exists(): load... else: fetch...`
If the file is already on disk, we skip the network request entirely. If the script crashes halfway through, you only restart from the failed city. This ensures fast, repeatable pipeline operations without redundant bandwidth consumption.


In [5]:
for key, cfg in CITIES_AQ.items():
    out_path = RAW_AQ / f'pm25_{key}_2025Q1.json'

    if out_path.exists():
        print(f'[{cfg["label"]}] file already exists – loading from disk.')
        with open(out_path) as f:
            data = json.load(f)
    else:
        print(f'[{cfg["label"]}] fetching from OpenAQ API …')
        data = fetch_openaq_daily(cfg['sensor_id'], DATE_FROM, DATE_TO)
        with open(out_path, 'w') as f:
            json.dump(data, f, indent=4)
        time.sleep(1)   # be polite to the API

    n = len(data.get('results', []))
    print(f'  → {n} daily records saved to {out_path.name}\n')

[New York City] file already exists – loading from disk.
  → 90 daily records saved to pm25_nyc_2025Q1.json

[Los Angeles] file already exists – loading from disk.
  → 90 daily records saved to pm25_la_2025Q1.json

[Chicago] file already exists – loading from disk.
  → 90 daily records saved to pm25_chicago_2025Q1.json



## 2. Open-Meteo Historical Weather API

We use the [Open-Meteo Historical API](https://archive-api.open-meteo.com/v1/archive) to fetch corresponding meteorological data, querying `temperature_2m`, `windspeed_10m`, and `precipitation`.

### Timezone Handling Mechanics
Unlike OpenAQ which natively provides UTC timestamps, we explicitly pass `timezone=auto` or a specific IANA timezone (e.g., `America/New_York`) to Open-Meteo. The API returns parallel arrays of data where the `time` array contains 'naive' string timestamps formatted in that exact local timezone. 

To align our weather data with our UTC-standardized Air Quality data later, we will have to dynamically parse these string arrays, attach the local timezone metadata, resolve any Daylight Saving Time ambiguities, and then convert back to UTC. We will tackle this in `02_clean_features.ipynb`.


In [6]:
OPENMETEO_URL = 'https://archive-api.open-meteo.com/v1/archive'

CITIES_WX = {
    'nyc':     {'lat': 40.7128, 'lon': -74.0060,  'label': 'New York City', 'tz': 'America/New_York'},
    'la':      {'lat': 34.0522, 'lon': -118.2437, 'label': 'Los Angeles',   'tz': 'America/Los_Angeles'},
    'chicago': {'lat': 41.8781, 'lon': -87.6298,  'label': 'Chicago',       'tz': 'America/Chicago'},
}

In [7]:
def fetch_openmeteo_hourly(lat: float, lon: float, tz: str,
                            date_from: str, date_to: str) -> dict:
    """Fetch hourly weather from the Open-Meteo Historical API."""
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'start_date': date_from,
        'end_date':   date_to,
        'hourly':     'temperature_2m,windspeed_10m,precipitation',
        'timezone':   tz,
    }
    resp = requests.get(OPENMETEO_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

### Idempotent Network Ingestion Loop (Weather)

Using the exact same idempotent pattern, we construct a parameterized `GET` request. We expect exactly 24 records per day for 90 days, yielding 2,160 rows of weather data per city. 


In [8]:
for key, cfg in CITIES_WX.items():
    out_path = RAW_WX / f'weather_{key}_2025Q1.json'

    if out_path.exists():
        print(f'[{cfg["label"]}] file already exists – loading from disk.')
        with open(out_path) as f:
            data = json.load(f)
    else:
        print(f'[{cfg["label"]}] fetching from Open-Meteo API …')
        data = fetch_openmeteo_hourly(
            cfg['lat'], cfg['lon'], cfg['tz'], DATE_FROM, DATE_TO
        )
        data['_city'] = key
        with open(out_path, 'w') as f:
            json.dump(data, f, indent=4)
        time.sleep(0.5)

    n = len(data.get('hourly', {}).get('time', []))
    print(f'  → {n} hourly records saved to {out_path.name}\n')

[New York City] file already exists – loading from disk.
  → 2160 hourly records saved to weather_nyc_2025Q1.json

[Los Angeles] file already exists – loading from disk.
  → 2160 hourly records saved to weather_la_2025Q1.json

[Chicago] file already exists – loading from disk.
  → 2160 hourly records saved to weather_chicago_2025Q1.json



## 3. Post-Ingestion Quality Check: Row Counts

Data engineering demands verification. We iterate over our fetched JSON arrays to verify their raw length. If a PM2.5 list returns 89 records instead of 90, we know immediately that the OpenAQ sensor was offline for a full day. This gives us immediate feedback on ingestion completeness before advancing to the transformation stage.


In [9]:
print('=== Air Quality (OpenAQ) – daily records per city ===')
for key, cfg in CITIES_AQ.items():
    path = RAW_AQ / f'pm25_{key}_2025Q1.json'
    with open(path) as f:
        d = json.load(f)
    n = len(d.get('results', []))
    print(f'  {cfg["label"]:20s} : {n:4d} rows')

print()
print('=== Weather (Open-Meteo) – hourly records per city ===')
for key, cfg in CITIES_WX.items():
    path = RAW_WX / f'weather_{key}_2025Q1.json'
    with open(path) as f:
        d = json.load(f)
    n = len(d.get('hourly', {}).get('time', []))
    print(f'  {cfg["label"]:20s} : {n:4d} rows')

=== Air Quality (OpenAQ) – daily records per city ===
  New York City        :   90 rows
  Los Angeles          :   90 rows
  Chicago              :   90 rows

=== Weather (Open-Meteo) – hourly records per city ===
  New York City        : 2160 rows
  Los Angeles          : 2160 rows
  Chicago              : 2160 rows
